# Eval-Driven Development for Enterprise AI Applications
## Workshop: Building a policy Q&A chatbot you can trust

---
### Table of Contents

[Part 0: Introduction & Setup](#introduction--setup)

[Part 1: Build the Policy Q&A Chatbot](#part-1-built-the-policy-qa-chatbot)
   - [1a. Load and chunk the policy documents](#1a-load-and-chunk-the-policy-documents)
   - [1b. Create the vector store](#1b-create-the-vector-store)
   - [1c. Build the RAG chain](#1c-build-the-rag-chain)

[Part 2: The Eval-Driven Development Mindset](#part-2-the-eval-driven-development-mindset)
   - [2a. What makes a good eval dataset?](#2a-what-makes-a-good-eval-dataset)

[Part 3: Inducing Predictable Failures](#part-3-inducing-predictable-failures)

[Part 4: Building the Evaluation Pipeline in LangSmith](#part-4-building-the-evaluation-pipeline-in-langsmith)
   - [4a. Upload the dataset to LangSmith](#4a-upload-the-dataset-to-langsmith)
   - [4b. Define Evaluators](#4b-define-evaluators)
   - [4c. Define the target function](#4c-define-the-target-function)
   - [4d. Run the eval experiments](#4d-run-the-eval-experiments)
   - [4e. Summarize and compare results](#4e-summarize-and-compare-results)

[Part 5: Human Review via Annotation Queues](#part-5-human-review-via-annotation-queues)
   - [5a. Create an annotation queue](#5a-create-an-annotation-queue)
   - [5b. Route low-scoring runs to the annotation queue](#5b-route-low-scoring-runs-to-the-annotation-queue)
   - [5c. The human reviewer's workflow](#5c-the-human-reviewers-workflow)

[Part 6: Iterating: When Do You Re-run Evals?](#part-6-iterating-when-do-you-re-run-evals)
   - [6a. Creating v2: prompt improvements](#6a-creating-v2-prompt-improvements)
   - [6b. Creating v3: changing model selection](#6b-creating-v3-changing-model-selection)

[Part 7: Eval Dataset Maintenance](#part-7-eval-dataset-maintenance)

[Summary and Key Takeaways](#summary-and-key-takeaways)

## Introduction & Setup

---
### The Problem

Enterprise teams building RAG-based applications often face a frustrating pattern:

1. Build a prototype → it answers questions, demos well → ship to pilot
2. A user finds a wrong answer → team fixes it → this causes something else to break
3. This cycle repeats, with no way to tell if the LLM's performance is actually improving

This happens because most teams build and iterate by reading outputs, tweaking prompts, and re-running the same few test questions. However, when your application has to handle more nuanced, context-dependent queries (different answers for different regions, roles, tenure levels), that approach collapses.

**Eval-driven development** (EDD) is the practice of treating evaluation as a first-class engineering discipline, not an afterthought. The core principle is to build a dataset of representative questions *before* you start tuning, define what "correct" looks like, and use that dataset as your ground truth every time you make a change.

### Today's Use Case: HR & IT Policy Q&A

HR and IT policy documents are a near-universal enterprise pain point:
- Every company has them, and they're always messy
- The *same question* has different correct answers depending on region, tenure, and role
- Errors aren't just unhelpful — they can be **legally or financially consequential**
- The failure modes are predictable and instructive

### What we will be building

1. A **RAG-based policy Q&A chatbot** over realistic HR and IT documents
2. A **golden eval dataset** with questions spanning easy lookups to multi-document cross-policy reasoning
3. A **LangSmith evaluation pipeline** with multiple evaluators (correctness, groundedness, refusal quality)
4. A **human review workflow** using LangSmith annotation queues
5. A disciplined practice around **when to re-run evals**

---

### Prerequisites
- Comfortable with Python
- Has built at least one LLM-powered app (RAG or agents)
- Basic understanding of LangSmith's purpose, features, and UI (no working experience necessary)
- LangChain API key
- LLM provider API key (OpenAI used in this workshop)
- Install dependencies via `requirements.txt`

In [8]:
# Install dependencies (uncomment and run if needed)
# ! pip3 install -r requirements.txt

In [5]:
import os
import json
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv

# load environment variables from .env file
load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY"
assert os.environ.get("LANGCHAIN_API_KEY"), "Set LANGCHAIN_API_KEY"

print("Environment ready ✓")
print(f"   LangSmith project: {os.environ.get('LANGCHAIN_PROJECT', 'default')}")
print(f"   Tracing enabled:   {os.environ.get('LANGCHAIN_TRACING_V2', 'false')}")

Environment ready ✓
   LangSmith project: policy-qa-eval-workshop
   Tracing enabled:   true


---

## Part 1: Build the Policy Q&A Chatbot

### 1a. Load and chunk the policy documents

In [6]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

POLICY_DIR = Path("policies")

# --- Load raw policy text ---
raw_docs = []
for policy_file in sorted(POLICY_DIR.glob("*.md")):
    loader = TextLoader(str(policy_file), encoding="utf-8")
    docs = loader.load()
    # Tag each document with its source file for citation later
    for doc in docs:
        doc.metadata["source"] = policy_file.name
        doc.metadata["policy_id"] = policy_file.stem
    raw_docs.extend(docs)
    print(f"  Loaded: {policy_file.name} ({len(docs[0].page_content):,} chars)")

print(f"\nTotal documents loaded: {len(raw_docs)}")

  Loaded: it_security_policy.md (5,658 chars)
  Loaded: parental_leave_policy.md (4,609 chars)
  Loaded: pto_policy.md (4,089 chars)

Total documents loaded: 3


In [7]:
# --- Split into chunks ---
splitter = RecursiveCharacterTextSplitter(
    # chunk_size=800,
    chunk_size=300,
    # chunk_overlap=100,
    chunk_overlap=0,
    separators=["\n## ", "\n### ", "\n#### ", "\n\n", "\n", " "],
)

chunks = splitter.split_documents(raw_docs)
print(f"Total chunks after splitting: {len(chunks)}")
print(f"\nSample chunk from '{chunks[5].metadata['source']}':")
print("-" * 60)
print(chunks[5].page_content[:400])
print("-" * 60)

Total chunks after splitting: 99

Sample chunk from 'it_security_policy.md':
------------------------------------------------------------
### Senior Manager and Director Level
- Directors and above may request a second company device (tablet or secondary laptop) for travel purposes. Requests must be approved by their VP.
- Directors have read access to the IT Security audit logs for their cost center.
------------------------------------------------------------


### 1b. Create the vector store

We're using FAISS (local, no external service needed for the workshop). In production, it is recommended to use a managed vector DB.

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

print("Embedding chunks... (this calls the OpenAI embeddings API)")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},  # Retrieve top 5 chunks
)

print(f"Vector store created with {vectorstore.index.ntotal} vectors ✓")

Embedding chunks... (this calls the OpenAI embeddings API)
Vector store created with 99 vectors ✓


### 1c. Build the RAG chain

In our V1 system prompt, we will include the following instructions:
- Citing the specific policy document and section
- Acknowledging when the answer depends on context (region, role, tenure)
- Refusing to answer if the policy doesn't cover the question

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- System prompt (v1 — we'll iterate on this) ---
SYSTEM_PROMPT_V1 = """
You are an HR and IT policy assistant for Acme Corp. Your job is to answer \
employee questions accurately based ONLY on the provided policy excerpts.

CRITICAL RULES:
1. Base your answer ONLY on the provided context. Do not use general knowledge \
about how companies typically handle these topics.
2. If the answer depends on the employee's region, role, or tenure, say so \
explicitly and provide the relevant details for each applicable variant.
3. If the context does not contain enough information to answer, say: \
"I don't have enough information in the current policy documents to answer this. Please contact HR or IT directly."
4. Cite the policy document and section (e.g., "per HR-PTO-001, Section 2").
5. Keep answers concise and factual. Avoid hedging language when the policy is clear.

Policy context:
---
{context}
---
"""

def format_docs(docs: list) -> str:
    """Format retrieved documents into a single context string."""
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        formatted.append(f"[Excerpt {i} — {source}]\n{doc.page_content}")
    return "\n\n".join(formatted)


llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.5)

prompt_v1 = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_V1),
    ("human", "{question}"),
])

# LCEL chain: retrieve → format → prompt → LLM → parse
policy_chain_v1 = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt_v1
    | llm
    | StrOutputParser()
)

print("RAG chain assembled (v1) ✓")

RAG chain assembled (v1) ✓


In [7]:
# Sample question
test_q = "How many PTO days do I get as a US employee with 2 years of tenure?"
response = policy_chain_v1.invoke(test_q)
print(f"Q: {test_q}\n")
print(f"A: {response}")

Q: How many PTO days do I get as a US employee with 2 years of tenure?

A: As a US employee with 2 years of tenure, you are entitled to 18 PTO days per year (per HR-PTO-001, Section 2).


---

## Part 2: The Eval-Driven Development Mindset

The LLM answers *something*. Now the critical question: **how do we know if it's actually correct?**

### Why only visual inspection of outputs doesn't scale

When you read one or two outputs and they look reasonable, you're doing **sample-based, subjective evaluation**. This is fine for catching catastrophic failures, but it:

- Doesn't tell you if you've *regressed* after a change
- Misses the long tail of hard cases
- Doesn't generalize — what you checked manually today isn't what will fail in production
- Doesn't give you a number to improve toward

### The EDD loop

```
Build dataset → Run eval → Read results → Change something → Re-run eval → Compare
     ↑                                                                          |
     └──────────────────────── (add new cases as you discover failures) ────────┘
```

The key insight: **the dataset is your contract**. It tells you, your teammates, and your stakeholders, exactly what the application is supposed to do. Changing something without running the eval is like merging code without running tests.

### 2a. What makes a good eval dataset?

A good eval dataset for a policy chatbot has:

| Property | Why it matters |
|----------|----------------|
| **Representative questions** | Covers the real distribution of what users will ask |
| **Varying difficulty** | Easy lookups + hard cross-policy reasoning |
| **Edge cases that encode known failure modes** | If you've seen it fail, it should be in the dataset |
| **Unambiguous ground truth** | You need a reference answer to evaluate against |
| **Refusal cases** | Questions the chatbot should *not* answer (or should hedge on) |

Let's look at our pre-built dataset:

In [8]:
# Load and inspect the eval dataset
with open("data/eval_dataset.json") as f:
    eval_cases = json.load(f)

print(f"Eval dataset: {len(eval_cases)} cases\n")

# Summarize by difficulty and tags
from collections import Counter

difficulty_counts = Counter(c["difficulty"] for c in eval_cases)
all_tags = [tag for c in eval_cases for tag in c["tags"]]
tag_counts = Counter(all_tags)

print("Difficulty distribution:")
for d, n in difficulty_counts.items():
    print(f"  {d:8s}: {n} cases")

print("\nTop topic tags:")
for tag, n in tag_counts.most_common(8):
    print(f"  {tag:20s}: {n}")

Eval dataset: 12 cases

Difficulty distribution:
  easy    : 3 cases
  medium  : 5 cases
  hard    : 4 cases

Top topic tags:
  pto                 : 5
  it                  : 4
  parental-leave      : 4
  us                  : 2
  accrual             : 2
  cross-policy        : 2
  california          : 1
  rollover            : 1


---

## Part 3: Inducing Predictable Failures

One of the hardest parts of teaching eval-driven development is demonstrating *why you need evals* — which means showing the application failing in instructive ways.

The problem: modern LLMs are very good at sounding correct even when they're wrong. If you just run the chatbot on hard questions and hope it fails, it often doesn't.

**The trick:** engineer the failure modes into the system deliberately, then show evals catching them. This is also realistic — you're not manufacturing fake bugs, you're simulating the kind of shortcuts real teams take.

We'll create two intentionally degraded versions:

### Failure Mode 1: Retrieval failure (bad `k` value selection, no overlap)
The retriever doesn't fetch enough context for multi-document questions.

### Failure Mode 2: Prompt that encourages hallucination
A prompt that doesn't anchor the model to the context, allowing it to fill gaps with general knowledge.

In [9]:
# --- Failure Mode 1: Stingy retriever (k=1, no overlap in chunking) ---
stingy_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},  # Only fetches 1 chunk — likely to miss context
)

# stingy retriever (k=1), decent prompt (v1)
bad_chain_retrieval = (
    {
        "context": stingy_retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt_v1
    | llm
    | StrOutputParser()
)

# --- Failure Mode 2: Hallucination-prone prompt ---
BAD_SYSTEM_PROMPT = """
You are an assistant for Acme Corp. Answer the employee's question about HR and IT policies. 

If the documents don't have a clear answer, use your best judgment about typical company policies.

Always provide an answer, even if the information is not in the provided context.

Context:
{context}
"""

prompt_bad = ChatPromptTemplate.from_messages([
    ("system", BAD_SYSTEM_PROMPT),
    ("human", "{question}"),
])

# better retriever (k=5), bad prompt
bad_chain_hallucination = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt_bad
    | llm
    | StrOutputParser()
)

print("Two degraded chains created for comparison ✓")

Two degraded chains created for comparison ✓


#### Now, let's start asking some questions!

**Note:** due to the non-deterministic nature of LLMs, our "bad" chains may actually produce some decent results. The important thing to remember here is even if we see a good response once, or even a few times, we are still lacking a standardized way to measure the quality of our LLM application.

In [10]:
# PTO question — requires combining two facts across the document
pto_question = "I'm a Dutch employee with 4 years of tenure. How many PTO days do I have, and when do they expire?"
pto_expected = "Employees in the Netherlands with 4 years of tenure get 30 days total (20 standard + 10 supplemental). \
The 20 standard days expire on July 1 of the following year, while the 10 supplemental days expire December 1 of the following year"

print("=" * 70)
print("QUESTION:", pto_question)
print("EXPECTED:", pto_expected)
print("=" * 70)

print("\n--- ✅ Good chain (v1) ---")
print(policy_chain_v1.invoke(pto_question))

print("\n--- ❌ Bad chain: stingy retriever (k=1) ---")
print(bad_chain_retrieval.invoke(pto_question))

print("\n--- ❌ Bad chain: hallucination-prone prompt ---")
print(bad_chain_hallucination.invoke(pto_question))

QUESTION: I'm a Dutch employee with 4 years of tenure. How many PTO days do I have, and when do they expire?
EXPECTED: Employees in the Netherlands with 4 years of tenure get 30 days total (20 standard + 10 supplemental). The 20 standard days expire on July 1 of the following year, while the 10 supplemental days expire December 1 of the following year

--- ✅ Good chain (v1) ---
As a Dutch employee with 4 years of tenure, you have a total of 30 PTO days (25 base days + 5 additional days after 3 years). Your PTO days expire as follows:
- Dutch statutory days (20) expire 6 months after the end of the accrual year (July 1 of the following year).
- Supplemental days (5) expire on December 31 of the following year (per Excerpt 2 and Excerpt 3).

--- ❌ Bad chain: stingy retriever (k=1) ---
Based on the provided policy excerpt, as a Dutch employee with 4 years of tenure, you are entitled to 27 PTO days per year. The policy document does not specify when these PTO days expire. I don't have enou

In [11]:
# Intentionally vague question, more information from user is required
pto_question = "How much supplemental parental leave do I get in the Netherlands?"

print("=" * 70)
print("QUESTION:", pto_question)
print("=" * 70)

print("\n--- ✅ Good chain (v1) ---")
print(policy_chain_v1.invoke(pto_question))

print("\n--- ❌ Bad chain: stingy retriever (k=1) ---")
print(bad_chain_retrieval.invoke(pto_question))

print("\n--- ❌ Bad chain: hallucination-prone prompt ---")
print(bad_chain_hallucination.invoke(pto_question))

QUESTION: How much supplemental parental leave do I get in the Netherlands?

--- ✅ Good chain (v1) ---
I don't have enough information in the current policy documents to answer this. Please contact HR or IT directly.

--- ❌ Bad chain: stingy retriever (k=1) ---
I don't have enough information in the current policy documents to answer this. Please contact HR or IT directly.

--- ❌ Bad chain: hallucination-prone prompt ---
In the Netherlands, Acme Corp provides supplemental parental leave as follows:
- For Partner (geboorteverlof): Acme tops up to 100% pay for 6 weeks (1 week at 100% pay from statutory + 5 weeks at 70% pay from statutory).
- For Adoptive/foster parent: Acme provides 6 weeks paid at 100%.
- For Parental leave (ouderschapsverlof): Acme does not supplement the 70% UWV portion, but the statutory entitlement is 26 weeks (9 weeks at 70% UWV).
- For Caregiver roles like Birth parent (bevallingsverlof): Acme tops up to 100% for the full 16 weeks of statutory leave.

If you have 

In [12]:
# Cross-policy question — requires reasoning across two separate documents
cross_question = "I'm going on parental leave in the US. I have 4 years of tenure. How many days of PTO will I accrue during my leave?"
cross_expected = "US employees with 4 years of tenure accrue PTO at a rate of ~1.83 days per month and will accrue PTO during paid parental leave. \
At 16 weeks of paid parental leave, you would accrue approximately 7.32 PTO days during your leave. (per HR-PTO-001, Section 3 and HR-Leave-002, Section 5)"

print("=" * 70)
print("QUESTION:", cross_question)
print("EXPECTED:", cross_expected)
print("=" * 70)

print("\n--- ✅ Good chain (v1) ---")
print(policy_chain_v1.invoke(cross_question))

print("\n--- ❌ Bad chain: stingy retriever (k=1) ---")
print(bad_chain_retrieval.invoke(cross_question))

print("\n--- ❌ Bad chain: hallucination-prone prompt ---")
print(bad_chain_hallucination.invoke(cross_question))

QUESTION: I'm going on parental leave in the US. I have 4 years of tenure. How many days of PTO will I accrue during my leave?
EXPECTED: US employees with 4 years of tenure accrue PTO at a rate of ~1.83 days per month and will accrue PTO during paid parental leave. At 16 weeks of paid parental leave, you would accrue approximately 7.32 PTO days during your leave. (per HR-PTO-001, Section 3 and HR-Leave-002, Section 5)

--- ✅ Good chain (v1) ---
During your parental leave in the US with 4 years of tenure, you will continue to accrue PTO at the rate of approximately 1.83 days per month, as per the standard accrual rate for employees with 3-5 years of tenure in the US (Excerpt 2 — pto_policy.md).

--- ❌ Bad chain: stingy retriever (k=1) ---
I don't have enough information in the current policy documents to answer this. Please contact HR or IT directly.

--- ❌ Bad chain: hallucination-prone prompt ---
During your parental leave in the US with 4 years of tenure, you will accrue PTO at a rat

---

## Part 4: Building the Evaluation Pipeline in LangSmith

Now we'll set up a proper evaluation pipeline in LangSmith. This covers:

1. Uploading our dataset to LangSmith
2. Defining custom and built-in evaluators
3. Running an eval experiments
4. Comparing results across different versions

### 4a. Upload the dataset to LangSmith

In [13]:
from langsmith import Client

ls_client = Client()

DATASET_NAME = "policy-qa-golden-v1"

# Check if dataset already exists
existing_datasets = list(ls_client.list_datasets(dataset_name=DATASET_NAME))

if existing_datasets:
    dataset = existing_datasets[0]
    print(f"Dataset already exists: {dataset.name} (id: {dataset.id})")
else:
    # Create the dataset
    dataset = ls_client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Golden eval dataset for Policy Q&A chatbot workshop. "
                    "12 questions covering PTO, IT Security, and Parental Leave policies "
                    "with regional, tenure, and role variation.",
    )

    # Upload examples
    examples_to_upload = [
        {
            "inputs": {"question": case["question"]},
            "outputs": {"expected_answer": case["expected_answer"]},
            "metadata": {
                "id": case["id"],
                "difficulty": case["difficulty"],
                "tags": case["tags"],
                "context": case["context"],
                "notes": case["notes"],
            },
        }
        for case in eval_cases
    ]

    ls_client.create_examples(
        inputs=[e["inputs"] for e in examples_to_upload],
        outputs=[e["outputs"] for e in examples_to_upload],
        metadata=[e["metadata"] for e in examples_to_upload],
        dataset_id=dataset.id,
    )

    print(f"Dataset created: {dataset.name} ✓")
    print(f"   ID: {dataset.id}")
    print(f"   {len(examples_to_upload)} examples uploaded")

Dataset created: policy-qa-golden-v1 ✓
   ID: f23eeb2e-7311-4d57-bcf0-cee9eb21be81
   12 examples uploaded


### 4b. Define Evaluators

This is the most important design decision in your eval pipeline. **Which evaluators you choose determines what you're optimizing for.**

For a policy Q&A chatbot, we care about:

| Evaluator | Question it answers | Why it matters |
|-----------|--------------------|-----------------|
| **Correctness** | Is the answer factually right vs. the ground truth? | Core accuracy metric |
| **Groundedness** | Does the answer stay within the provided context? | Catches hallucination |
| **Completeness** | Did the chatbot cover all relevant aspects? | Catches partial answers |
| **Conciseness** | Is the answer appropriately brief? | Long answers cause skimming — employees misread policy details buried in walls of text |
| **Built-in QA** | Does the answer correctly address the question? (no custom rubric needed) | Quick sanity check; useful to compare against your custom correctness evaluator |

We'll implement the first four as custom LLM-as-judge evaluators, and the fifth using LangSmith's built-in QA evaluator — so you can see both approaches side by side.

> **When to use LLM-as-a-judge vs. exact match vs. human review:**
> - **Exact match:** Only when answers are strictly deterministic (IDs, numbers, yes/no). Fragile for prose.
> - **LLM-as-a-judge:** Good for open-ended text when you can write a clear rubric. Scales well. Not perfect — the judge can also be wrong.
> - **Human review:** Ground truth, but expensive. Use for your hardest cases and to periodically calibrate your LLM judge.

In [14]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate
from langsmith.schemas import Run, Example

JUDGING_MODEL = "gpt-4o" # Using a stronger model for evaluation to improve judgment quality

# ── Evaluator 1: Correctness (LLM-as-a-judge, custom) ─────────────────────────────────

CORRECTNESS_PROMPT = """
You are an expert evaluator for an HR and IT policy Q&A system.

Your task: Determine whether the [Answer] correctly addresses the [Question] \
given the [Reference Answer].

Question: {question}
Reference Answer: {reference}
Answer: {prediction}

Scoring rubric:
- Score 1 (Correct): The answer captures all key facts from the reference answer \
and contains no factual errors. Minor phrasing differences are fine.
- Score 0.5 (Partially Correct): The answer is partially right but missing \
important details, OR contains one factual error alongside correct information.
- Score 0 (Incorrect): The answer contains significant factual errors, \
contradicts the reference, or fails to address the key question.

Respond ONLY in this JSON format (no other text):
{{"score": <0 | 0.5 | 1>, "reasoning": "<one sentence explaining your score>"}}
"""

def correctness_evaluator(run: Run, example: Example) -> dict:
    """LLM-as-judge correctness evaluator."""
    question = example.inputs.get("question", "")
    reference = example.outputs.get("expected_answer", "")
    prediction = run.outputs.get("output", "") if run.outputs else ""

    judge_llm = ChatOpenAI(model=JUDGING_MODEL, temperature=0)
    prompt = CORRECTNESS_PROMPT.format(
        question=question, reference=reference, prediction=prediction
    )
    result = judge_llm.invoke(prompt).content

    try:
        parsed = json.loads(result)
        score = float(parsed["score"])
        reasoning = parsed["reasoning"]
    except (json.JSONDecodeError, KeyError):
        score = 0.0
        reasoning = f"Failed to parse judge response: {result}"

    return {"key": "correctness", "score": score, "comment": reasoning}

In [15]:
# ── Evaluator 2: Groundedness (LLM-as-a-judge, custom) ────────────────────────────────────────────────

GROUNDEDNESS_PROMPT = """
You are evaluating whether an AI assistant's answer is grounded in the provided \
policy documents, or whether it introduces information not present in the context.

Question: {question}
Answer: {prediction}

Evaluate whether every specific factual claim in the answer (numbers, dates, \
rules, exceptions) is something that could plausibly come from an HR/IT policy \
document, or whether the answer appears to include "filler" knowledge not tied \
to any specific policy.

Note: We do NOT have the actual retrieved context here — evaluate based on \
whether the answer is appropriately hedged and policy-specific, vs. generic.

Scoring:
- Score 1: All claims appear specific and policy-grounded; no generic company advice
- Score 0.5: Mix of specific policy claims and generic advice
- Score 0: Answer relies heavily on generic knowledge; not grounded in specific policy

Respond ONLY in JSON: {{"score": <0 | 0.5 | 1>, "reasoning": "<one sentence>"}}
"""

def groundedness_evaluator(run: Run, example: Example) -> dict:
    """Evaluates whether the answer is grounded in policy content vs. generic knowledge."""
    question = example.inputs.get("question", "")
    prediction = run.outputs.get("output", "") if run.outputs else ""

    judge_llm = ChatOpenAI(model=JUDGING_MODEL, temperature=0)
    prompt = GROUNDEDNESS_PROMPT.format(question=question, prediction=prediction)
    result = judge_llm.invoke(prompt).content

    try:
        parsed = json.loads(result)
        score = float(parsed["score"])
        reasoning = parsed["reasoning"]
    except (json.JSONDecodeError, KeyError):
        score = 0.0
        reasoning = f"Parse error: {result}"

    return {"key": "groundedness", "score": score, "comment": reasoning}

In [37]:
# ── Evaluator 3: Completeness (LLM-as-a-judge, custom) ────────────────────────────────────────────────

COMPLETENESS_PROMPT = """
You are evaluating whether an AI assistant gave a COMPLETE answer to a policy question.

Question: {question}
Reference Answer (covers all key points): {reference}
Actual Answer: {prediction}

Identify which key facts from the reference answer are MISSING from the actual answer.

Scoring:
- Score 1: No meaningful facts are missing
- Score 0.5: 1-2 important facts are missing
- Score 0: Most important facts are missing or the answer is substantially incomplete

Respond ONLY in JSON: {{"score": <0 | 0.5 | 1>, "missing_facts": ["..."], "reasoning": "<one sentence>"}}
"""

def completeness_evaluator(run: Run, example: Example) -> dict:
    """Evaluates whether the answer covers all key points from the reference."""
    question = example.inputs.get("question", "")
    reference = example.outputs.get("expected_answer", "")
    prediction = run.outputs.get("output", "") if run.outputs else ""

    judge_llm = ChatOpenAI(model=JUDGING_MODEL, temperature=0)
    prompt = COMPLETENESS_PROMPT.format(
        question=question, reference=reference, prediction=prediction
    )
    result = judge_llm.invoke(prompt).content

    try:
        parsed = json.loads(result)
        score = float(parsed["score"])
        reasoning = parsed.get("reasoning", "")
        missing = parsed.get("missing_facts", [])
        comment = reasoning
        if missing:
            comment += f" Missing: {missing}"
    except (json.JSONDecodeError, KeyError):
        score = 0.0
        comment = f"Parse error: {result}"

    return {"key": "completeness", "score": score, "comment": comment}

In [17]:
# ── Evaluator 4: Conciseness (LLM-as-a-judge, custom) ──────────────────────────
# Policy answers that are too long are a real safety risk: employees skim them
# and miss critical details (e.g., the one sentence that says "except in California").

CONCISENESS_PROMPT = """
You are evaluating whether an AI assistant gave a CONCISE answer to a policy question.

Policy answers that are too long are risky: employees skim them and may act on \
partial information or miss important exceptions.

Question: {question}
Answer: {prediction}

Scoring:
- Score 1: Appropriately concise — covers key facts without unnecessary elaboration, \
hedging, or repetition. Typically 1-4 sentences for simple questions, up to a short \
paragraph for multi-part or cross-policy questions.
- Score 0.5: Somewhat verbose — includes some unnecessary elaboration but key \
information is still accessible.
- Score 0: Excessively long or padded — buries key facts in hedging, repetition, or \
filler that would cause a reader to skim and miss critical information.

Respond ONLY in JSON: {{"score": <0 | 0.5 | 1>, "reasoning": "<one sentence>"}}
"""

def conciseness_evaluator(run: Run, example: Example) -> dict:
    """Evaluates whether the answer is appropriately concise for policy communication."""
    question = example.inputs.get("question", "")
    prediction = run.outputs.get("output", "") if run.outputs else ""

    judge_llm = ChatOpenAI(model=JUDGING_MODEL, temperature=0)
    prompt = CONCISENESS_PROMPT.format(question=question, prediction=prediction)
    result = judge_llm.invoke(prompt).content

    try:
        parsed = json.loads(result)
        score = float(parsed["score"])
        reasoning = parsed["reasoning"]
    except (json.JSONDecodeError, KeyError):
        score = 0.0
        reasoning = f"Parse error: {result}"

    return {"key": "conciseness", "score": score, "comment": reasoning}

In [18]:
# ── Evaluator 5: Built-in QA evaluator (LLM-as-a-judge, built-in) ──────────────────────────────────────
# LangSmith's LangChainStringEvaluator wraps LangChain's built-in evaluators.
# "qa" checks whether the prediction correctly answers the question given a reference,
# using a fixed internal prompt — no rubric required from us.

builtin_qa_evaluator = LangChainStringEvaluator(
    "qa",
    config={"llm": ChatOpenAI(model=JUDGING_MODEL, temperature=0)},
    prepare_data=lambda run, example: {
        "prediction": run.outputs.get("output", "") if run.outputs else "",
        "input": example.inputs.get("question", ""),
        "reference": example.outputs.get("expected_answer", ""),
    },
)

In [19]:
ALL_EVALUATORS = [
    correctness_evaluator,
    groundedness_evaluator,
    completeness_evaluator,
    conciseness_evaluator,
    builtin_qa_evaluator,
]

print("Evaluators defined:")
for ev in ALL_EVALUATORS:
    name = ev.__name__ if callable(ev) and hasattr(ev, "__name__") else type(ev).__name__
    print(f"   - {name}")

Evaluators defined:
   - correctness_evaluator
   - groundedness_evaluator
   - completeness_evaluator
   - conciseness_evaluator
   - LangChainStringEvaluator


> **💬 Discussion: Choosing the right evaluators**
>
> For a policy chatbot, **correctness** is the headline metric, but it alone can be misleading:
> - A hallucinated answer that happens to match the reference scores high on correctness
> - A correct but incomplete answer scores high on correctness but misses key information
>
> That's why we run **groundedness** and **completeness** in parallel. They catch different failure modes.
>
> **Conciseness** is easy to overlook, but it's particularly important for policy Q&A. A chatbot that gives a technically correct but 300-word answer to a simple question creates a real risk: the employee skims it, misses the exception clause, and acts on incomplete information. Verbosity and accuracy are independent failure modes — an answer can be both correct and dangerous.
>
> A question worth asking for every evaluator you add: *"What specific failure mode does this catch that the others don't?"* If you can't answer that, the evaluator may be redundant.

---

> **💬 Built-in vs. custom evaluators: when to use each**
>
> The built-in `"qa"` evaluator and our custom `correctness_evaluator` are measuring roughly the same thing — whether the answer is factually right. Running both is useful here to calibrate them against each other, but in production you'd pick one.
>
> | | **Built-in evaluator** | **Custom LLM-as-judge** |
> |---|---|---|
> | **Setup** | One line, no prompt writing | Requires a rubric you write and maintain |
> | **Transparency** | Opaque — you don't control the scoring logic | Full visibility — you can read, test, and tune the prompt |
> | **Domain fit** | Generic; works across use cases | Tailored to your domain (e.g., "per policy section X" is meaningful here) |
> | **Consistency** | Stable across LangChain versions (mostly) | Can drift if you change the prompt |
> | **Debugging** | Hard to diagnose when scores look wrong | Easy — inspect the prompt and the judge's reasoning |
>
> **Rule of thumb:** Start with a built-in evaluator to get a baseline quickly. Once you find cases where it scores incorrectly for your domain, write a custom evaluator with an explicit rubric.
>
> You'll often end up running both: the built-in as a low-overhead sanity check, and the custom one as your primary signal.

### 4c. Define the target function

LangSmith's `evaluate()` function needs a callable that takes dataset inputs and returns outputs. We'll wrap our chains.

In [20]:
def make_target(chain):
    """Wrap a chain into a LangSmith-compatible target function."""
    def target(inputs: dict) -> dict:
        question = inputs["question"]
        output = chain.invoke(question)
        return {"output": output}
    return target


target_v1 = make_target(policy_chain_v1)
target_bad_retrieval = make_target(bad_chain_retrieval)
target_bad_hallucination = make_target(bad_chain_hallucination)

print("Target functions ready ✓")

Target functions ready ✓


### 4d. Run the eval experiments

We'll run all three chain versions against the same dataset. Each run creates an **experiment** in LangSmith that you can inspect, filter, and compare in the UI.

> ⏱ This makes LLM API calls for every example × every evaluator. With 12 examples and 5 evaluators, expect ~60 additional LLM calls per chain run. Budget ~1-3 minutes per chain.

In [21]:
# Run experiment for the good chain (v1)
print("Running eval for chain v1 (good baseline)...")
results_v1 = evaluate(
    target_v1,
    data=DATASET_NAME,
    evaluators=ALL_EVALUATORS,
    experiment_prefix="policy-qa-v1-baseline",
    metadata={"chain_version": "v1", "retriever_k": 5, "prompt": "v1"},
)
print("v1 baseline complete ✓")

Running eval for chain v1 (good baseline)...
View the evaluation results for experiment: 'policy-qa-v1-baseline-4cf3b092' at:
https://smith.langchain.com/o/064ff6ad-153f-49c0-ac87-3909a1bbdeaa/datasets/f23eeb2e-7311-4d57-bcf0-cee9eb21be81/compare?selectedSessions=586f4a12-5b5e-433a-b8ba-90319f3e2d1d




0it [00:00, ?it/s]

v1 baseline complete ✓


In [22]:
# Run experiment for the bad retrieval chain
print("Running eval for bad_retrieval (k=1)...")
results_bad_retrieval = evaluate(
    target_bad_retrieval,
    data=DATASET_NAME,
    evaluators=ALL_EVALUATORS,
    experiment_prefix="policy-qa-bad-retrieval",
    metadata={"chain_version": "bad_retrieval", "retriever_k": 1, "prompt": "v1"},
)
print("bad_retrieval eval complete ✓")

Running eval for bad_retrieval (k=1)...
View the evaluation results for experiment: 'policy-qa-bad-retrieval-ecc9feca' at:
https://smith.langchain.com/o/064ff6ad-153f-49c0-ac87-3909a1bbdeaa/datasets/f23eeb2e-7311-4d57-bcf0-cee9eb21be81/compare?selectedSessions=14152dd4-8bb2-4457-beb2-1bb9d5a9d5ce




0it [00:00, ?it/s]

bad_retrieval eval complete ✓


In [23]:
# Run experiment for the hallucination-prone chain
print("Running eval for bad_hallucination...")
results_bad_hallucination = evaluate(
    target_bad_hallucination,
    data=DATASET_NAME,
    evaluators=ALL_EVALUATORS,
    experiment_prefix="policy-qa-bad-hallucination",
    metadata={"chain_version": "bad_hallucination", "retriever_k": 5, "prompt": "bad_v1"},
)
print("bad_hallucination eval complete ✓")

Running eval for bad_hallucination...
View the evaluation results for experiment: 'policy-qa-bad-hallucination-17baab6f' at:
https://smith.langchain.com/o/064ff6ad-153f-49c0-ac87-3909a1bbdeaa/datasets/f23eeb2e-7311-4d57-bcf0-cee9eb21be81/compare?selectedSessions=0362569b-0a73-4dc4-a1ed-6e1e67724925




0it [00:00, ?it/s]

bad_hallucination eval complete ✓


### 4e. Summarize and compare results

In [24]:
import statistics

def summarize_results(results, label: str) -> dict:
    """Extract mean scores per evaluator from a LangSmith evaluate() result."""
    scores_by_key = {}
    for result in results._results:
        for fb in result.get("evaluation_results", {}).get("results", []):
            key = fb.key
            score = fb.score
            if score is not None:
                scores_by_key.setdefault(key, []).append(score)

    summary = {"label": label}
    for key, scores in scores_by_key.items():
        summary[key] = round(statistics.mean(scores), 3)
    return summary


summaries = [
    summarize_results(results_v1, "v1 (good baseline)"),
    summarize_results(results_bad_retrieval, "bad_retrieval (k=1)"),
    summarize_results(results_bad_hallucination, "bad_hallucination"),
]

# Print a comparison table
metrics = ["correctness", "groundedness", "completeness", "conciseness"]
df = pd.DataFrame([
    {"Chain": s["label"], **{m: s.get(m) for m in metrics}}
    for s in summaries
]).set_index("Chain")

df

,correctness,groundedness,completeness,conciseness
Chain,,,,
v1 (good baseline),0.771,0.792,0.0,0.000
bad_retrieval (k=1),0.750,0.958,0.0,0.167
bad_hallucination,0.875,0.375,0.0,0.042


> **💬 Reading the results table**
>
> In most cases, you should see:
> - `bad_retrieval` scoring lowest on **completeness** — it is unable to answer multi-fact questions with only 1 chunk
> - `bad_hallucination` scoring lowest on **groundedness** — it fills gaps with generic knowledge
> - `v1` should score best on correctness overall, though it may still slip on the hardest cross-policy cases
>
> The important thing: **these differences are systematic**, not just one bad answer. That's what evals tell you that manual inspection doesn't.

---

## Part 5: Human Review via Annotation Queues

LLM-as-a-judge evaluators are powerful, but they're not infallible. For high-stakes policy questions, you want a human review step — especially for:

- Cases where the judge gave a borderline score (0.5)
- New failure modes your evaluators weren't designed to catch
- Calibrating whether your LLM judge itself is well-calibrated

LangSmith's **annotation queues** give you a structured workflow for human review that integrates directly with your eval results.

### When should you route to human review?

| Trigger | Why |
|---------|-----|
| Correctness score ≤ 0.5 | Potential wrong answer — needs human confirmation |
| Groundedness score < 1.0 | May have hallucinated — human should check against raw policy |
| Hard-difficulty cases | These have nuance LLM judges may miss |
| Refusal cases | Is the hedge appropriate? Hard for LLM to judge |
| Any case flagged as `cross-policy` | Requires synthesis the judge may undervalue |

### 5a. Create an annotation queue

In [25]:
# Create an annotation queue in LangSmith
QUEUE_NAME = "policy-qa-human-review"

try:
    queue = ls_client.create_annotation_queue(
        name=QUEUE_NAME,
        description="Human review queue for policy Q&A chatbot. "
                    "Routes borderline and hard cases for expert review.",
    )
    print(f"Annotation queue created: {queue.name} (id: {queue.id})")
except Exception as e:
    # Queue may already exist
    queues = list(ls_client.list_annotation_queues(name=QUEUE_NAME))
    if queues:
        queue = queues[0]
        print(f"Using existing annotation queue: {queue.name} (id: {queue.id})")
    else:
        raise e

Annotation queue created: policy-qa-human-review (id: ebb9e169-55fb-4d33-90f4-b5043e06867b)


### 5b. Route low-scoring runs to the annotation queue

Strategy: 
- any run where correctness < 1.0 OR 
- the example is "hard" difficulty

In [26]:
# Find runs that need human review

hard_ids = {c["id"] for c in eval_cases if c["difficulty"] == "hard"}

runs_to_review = []

for result in results_v1._results:
    run = result.get("run")
    example = result.get("example")
    if run is None or example is None:
        continue

    # Get scores
    eval_results = result.get("evaluation_results", {}).get("results", [])
    scores = {fb.key: fb.score for fb in eval_results if fb.score is not None}

    correctness = scores.get("correctness", 1.0)
    example_id = example.metadata.get("id", "") if example.metadata else ""
    difficulty = example.metadata.get("difficulty", "") if example.metadata else ""

    should_review = (
        correctness < 1.0
        or difficulty == "hard"
        or example_id in hard_ids
    )

    if should_review:
        runs_to_review.append({
            "run_id": str(run.id),
            "example_id": example_id,
            "correctness": correctness,
            "difficulty": difficulty,
            "reason": "hard case" if difficulty == "hard" else "low correctness",
        })

print(f"Found {len(runs_to_review)} runs to route to human review")
for r in runs_to_review:
    print(f"   {r['example_id']:15s} | correctness={r['correctness']} | reason: {r['reason']}")

Found 5 runs to route to human review
   cross-001       | correctness=1 | reason: hard case
   pto-003         | correctness=1 | reason: hard case
   leave-002       | correctness=1 | reason: hard case
   it-003          | correctness=0 | reason: low correctness
   cross-002       | correctness=1 | reason: hard case


In [27]:
# Add runs to the annotation queue
run_ids_to_add = [r["run_id"] for r in runs_to_review]

if run_ids_to_add:
    ls_client.add_runs_to_annotation_queue(
        queue_id=queue.id,
        run_ids=run_ids_to_add,
    )
    print(f"Added {len(run_ids_to_add)} runs to annotation queue '{QUEUE_NAME}'")
    print(f"   Open in LangSmith UI to review: https://smith.langchain.com")
else:
    print("No runs needed routing (all scored 1.0 on correctness)")

Added 5 runs to annotation queue 'policy-qa-human-review'
   Open in LangSmith UI to review: https://smith.langchain.com


### 5c. The human reviewer's workflow

In the LangSmith UI, a human reviewer navigating to the annotation queue will see:

1. The original question
2. The chatbot's answer
3. The expected answer (reference)
4. The automated eval scores and reasoning

They can then:
- **Confirm or override** the automated score
- **Add a note** explaining the issue (e.g., "missed the California-specific rule")
- **Flag for prompt engineering** or **flag for dataset gap**

> **💬 Discussion: What do you do with human review feedback?**
>
> Human review creates two types of signal:
> 1. **The chatbot is wrong → fix it**: diagnose whether it's a retrieval problem, a prompt problem, or a knowledge gap in the documents themselves
> 2. **The evaluator is wrong → fix it**: if human reviewers consistently disagree with your LLM judge, your judge's rubric needs updating
>
> Both are valuable. The second one is often overlooked — evaluator quality is a first-class engineering concern.

---

## Part 6: Iterating: When Do You Re-run Evals?

This is one of the most important discipline questions in eval-driven development. Running evals costs time and money, so you can't run them after every line change. But if you don't run them often enough, you won't catch regressions.

#### The rule of thumb: **Run evals whenever you change something that could affect output quality**

| Change | Should you re-run evals? | Notes |
|--------|--------------------------|-------|
| Changing the system prompt | **Yes, always** | Even small prompt changes can have large, unexpected effects |
| Changing retriever `k` | **Yes** | Directly affects what context is available |
| Changing chunk size or overlap | **Yes** | Affects which information ends up in the same chunk |
| Switching embedding model | **Yes** | Changes retrieval quality across the board |
| Switching LLM model/version | **Yes** | Different models have different failure modes |
| Adding new policy documents | **Yes** | May introduce retrieval conflicts or improve coverage |
| Fixing a bug in the formatting logic | **Maybe** | Only if it changes the context fed to the LLM |
| Updating logging or metadata | **No** | Doesn't affect outputs |
| Refactoring without behavior change | **No, but consider a smoke test** | |

### 6a. Creating v2: prompt improvements

Let's make a targeted improvement to address one specific failure pattern we observed: the chatbot sometimes gives the California rule when answering general US questions. We'll add an explicit instruction, then re-run.

In [28]:
# --- System prompt v2: Adds explicit instructions for context-dependent answers ---
SYSTEM_PROMPT_V2 = """
You are an HR and IT policy assistant for Acme Corp. Your job is to answer \
employee questions accurately based ONLY on the provided policy excerpts.

CRITICAL RULES:
1. Base your answer ONLY on the provided context. Do not use general knowledge.
2. CONTEXT-DEPENDENT ANSWERS: If the answer varies by region, role, or tenure:
   a. Always lead with the most specific answer for the employee's stated context.
   b. Note if other regions/roles have different rules only if directly relevant.
   c. Do NOT default to the most common case if the employee has stated their context.
   d. Note that some policies may have exceptions (e.g., "except in California") — these are critical to include if present.
3. CROSS-POLICY QUESTIONS: If the question requires combining information from \
multiple policies (e.g., how parental leave interacts with PTO), explicitly \
identify both policies and synthesize them.
4. UNCERTAINTY: If the context does not contain enough information, say: \
"I don't have enough information in the current policy documents. Please contact \
HR or IT directly."
5. Always cite the policy document and section (e.g., "per HR-PTO-001, Section 2").
6. Keep answers concise and factual.

Policy context:
---
{context}
---
"""

prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_V2),
    ("human", "{question}"),
])

policy_chain_v2 = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt_v2
    | llm
    | StrOutputParser()
)

target_v2 = make_target(policy_chain_v2)
print("Chain v2 assembled with improved prompt ✓")

Chain v2 assembled with improved prompt ✓


In [29]:
# Run evals on v2
print("Running eval for chain v2 (improved prompt)...")
results_v2 = evaluate(
    target_v2,
    data=DATASET_NAME,
    evaluators=ALL_EVALUATORS,
    experiment_prefix="policy-qa-v2-improved-prompt",
    metadata={"chain_version": "v2", "retriever_k": 5, "prompt": "v2"},
)
print("v2 eval complete ✓")

Running eval for chain v2 (improved prompt)...
View the evaluation results for experiment: 'policy-qa-v2-improved-prompt-e5aa7f76' at:
https://smith.langchain.com/o/064ff6ad-153f-49c0-ac87-3909a1bbdeaa/datasets/f23eeb2e-7311-4d57-bcf0-cee9eb21be81/compare?selectedSessions=93ccfecb-d61e-4b24-9e02-0de4cdeabcb3




0it [00:00, ?it/s]

v2 eval complete ✓


### 6b. Creating v3: changing model selection

In [30]:
llm_v2 = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

policy_chain_v3 = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt_v2
    | llm_v2
    | StrOutputParser()
)

target_v3 = make_target(policy_chain_v3)
print("v3 assembled with new model ✓")

v3 assembled with new model ✓


In [31]:
# Run evals on v3
print("Running eval for chain v3 (new model, gpt-4o-mini)...")
results_v3 = evaluate(
    target_v3,
    data=DATASET_NAME,
    evaluators=ALL_EVALUATORS,
    experiment_prefix="policy-qa-v3-improved-prompt",
    metadata={"chain_version": "v3", "retriever_k": 5, "prompt": "v2"},
)
print("v3 eval complete ✓")

Running eval for chain v3 (new model, gpt-4o-mini)...
View the evaluation results for experiment: 'policy-qa-v3-improved-prompt-bd2b5563' at:
https://smith.langchain.com/o/064ff6ad-153f-49c0-ac87-3909a1bbdeaa/datasets/f23eeb2e-7311-4d57-bcf0-cee9eb21be81/compare?selectedSessions=b3f5e629-8f38-4a56-b912-65ff05a47464




0it [00:00, ?it/s]

v3 eval complete ✓


In [33]:
# Final comparison: all four experiments
all_summaries = [
    summarize_results(results_v1, "v1 baseline"),
    summarize_results(results_bad_retrieval, "bad_retrieval (k=1)"),
    summarize_results(results_bad_hallucination, "bad_hallucination"),
    summarize_results(results_v2, "v2 improved prompt"),
    summarize_results(results_v3, "v3 improved prompt + gpt-4o"),
]

metrics = ["correctness", "groundedness", "completeness", "conciseness"]
df = pd.DataFrame([
    {"Chain": s["label"], **{m: s.get(m) for m in metrics}}
    for s in all_summaries
]).set_index("Chain")

df

,correctness,groundedness,completeness,conciseness
Chain,,,,
v1 baseline,0.771,0.792,0.0,0.000
bad_retrieval (k=1),0.750,0.958,0.0,0.167
bad_hallucination,0.875,0.375,0.0,0.042
v2 improved prompt,0.854,0.792,0.0,0.167
v3 improved prompt + gpt-4o,0.729,0.667,0.0,0.125


> **💬 Discussion: How do you know if a change is "good enough"?**
>
> There's no universal threshold. Some heuristics:
> - **A regression on any metric is a red flag** — understand why before shipping
> - For high-stakes contexts (policy, legal, medical), aim for correctness ≥ 0.9
> - Track scores over time and set a **minimum bar** for production deployment (e.g., "no release if correctness < 0.85")
> - Pay attention to **hard cases specifically** — aggregate metrics can hide poor performance on your most important queries
>
> Also remember: **eval scores are not ground truth**. They're a signal. Always sanity-check score changes against actual outputs before concluding that an improvement is real.

---

## Part 7: Eval Dataset Maintenance

Eval datasets are living assets that require maintenance over time.

### When to add new cases

| Trigger | Action |
|---------|--------|
| A user reports a wrong answer | Add the question + correct answer to the dataset immediately |
| A new policy document is added | Add cases covering the new policy's key rules and edge cases |
| You discover a new failure mode during manual review | Add cases that encode the failure mode |
| You add a new user segment (e.g., new country) | Add cases covering that segment's specific rules |

### Dataset anti-patterns to avoid

| Anti-pattern | Why it's dangerous |
|-------------|--------------------|
| **Teaching to the test** | Optimizing your prompt specifically for eval cases → evals stop being predictive |
| **Stale ground truth** | Policy changes but eval dataset doesn't update → correct answers start scoring as wrong |
| **Only easy cases** | High scores look great, but you're not testing the hard stuff |
| **No refusal cases** | Chatbot has no incentive to say "I don't know" → overconfident on out-of-scope questions |
| **One question per topic** | If your one Netherlands PTO question happens to be easy, you have a blind spot |

### Adding a new case from a real failure

In [34]:
# Simulate: a user reported a wrong answer about PTO during unpaid leave
# We discovered the chatbot confidently stated PTO accrues during all types of leave.
# This is a new edge case not in our original dataset.

new_case = {
    "inputs": {
        "question": "I'm taking unpaid leave for 3 months for a personal project. Will my PTO keep accruing?"
    },
    "outputs": {
        "expected_answer": "No. Per HR-PTO-001, Section 4, employees on approved unpaid leave do not accrue PTO. PTO accrual only continues during paid leave."
    },
    "metadata": {
        "id": "pto-005",
        "difficulty": "medium",
        "tags": ["pto", "unpaid-leave", "accrual", "edge-case"],
        "context": {"region": "US", "tenure_years": 2, "role": "Individual Contributor"},
        "notes": "Added after user report. Chatbot was incorrectly saying PTO accrues during unpaid leave.",
        "source": "user_report",
    },
}

ls_client.create_examples(
    inputs=[new_case["inputs"]],
    outputs=[new_case["outputs"]],
    metadata=[new_case["metadata"]],
    dataset_id=dataset.id,
)

print("New case added to dataset from user report")
print(f"   Dataset '{DATASET_NAME}' now has {len(eval_cases) + 1} examples")
print("\n💡 Best practice: after adding new cases, re-run evals against the updated dataset")
print("   to establish the new baseline before your next code change.")

New case added to dataset from user report
   Dataset 'policy-qa-golden-v1' now has 13 examples

💡 Best practice: after adding new cases, re-run evals against the updated dataset
   to establish the new baseline before your next code change.


In [35]:
# Run evals on v3 again with the updated dataset (including the new user-reported case)
print("Running eval for chain v3 (new model, gpt-4o)...")
results_v3_update = evaluate(
    target_v3,
    data=DATASET_NAME,
    evaluators=ALL_EVALUATORS,
    experiment_prefix="policy-qa-v3-improved-prompt",
    metadata={"chain_version": "v3", "retriever_k": 5, "prompt": "v2"},
)
print("updated v3 eval complete ✓")

Running eval for chain v3 (new model, gpt-4o)...
View the evaluation results for experiment: 'policy-qa-v3-improved-prompt-e7d8ef1b' at:
https://smith.langchain.com/o/064ff6ad-153f-49c0-ac87-3909a1bbdeaa/datasets/f23eeb2e-7311-4d57-bcf0-cee9eb21be81/compare?selectedSessions=e81b21a2-f963-4c2c-a873-a63613e95cbc




0it [00:00, ?it/s]

updated v3 eval complete ✓


In [36]:
all_summaries = [
    # summarize_results(results_v1, "v1 baseline"),
    # summarize_results(results_bad_retrieval, "bad_retrieval (k=1)"),
    # summarize_results(results_bad_hallucination, "bad_hallucination"),
    # summarize_results(results_v2, "v2 improved prompt"),
    summarize_results(results_v3, "v3 improved prompt + gpt-4o"),
    summarize_results(results_v3_update, "v3, user-reported case added"),
]

metrics = ["correctness", "groundedness", "completeness", "conciseness"]
df = pd.DataFrame([
    {"Chain": s["label"], **{m: s.get(m) for m in metrics}}
    for s in all_summaries
]).set_index("Chain")

df

,correctness,groundedness,completeness,conciseness
Chain,,,,
v3 improved prompt + gpt-4o,0.729,0.667,0.0,0.125
"v3, user-reported case added",0.750,0.731,0.0,0.154


---

## Summary and Key Takeaways

### What we built

1. **A RAG-based policy Q&A chatbot** that handles complex, context-dependent queries across HR and IT documents
2. **A golden eval dataset** with 12 curated cases spanning easy lookups to cross-document reasoning
3. **Three custom LLM-as-a-judge evaluators** measuring correctness, groundedness, and completeness
4. **A LangSmith evaluation pipeline** that compares iapplication versions systematically
5. **A human annotation queue** for routing borderline and high-stakes cases to expert review
6. **A v2 and v3 improvement** with measurable score changes you can trace in the LangSmith UI



### The core principle

> **Treat your eval dataset like a test suite. It's not optional. It's the contract.**

You wouldn't merge code without running tests. The same discipline applies to developing LLM-based applications:
- Changes to prompts, retrieval, or models → re-run evals
- Real failures in production → add to the dataset before fixing
- Evals that stop being predictive → improve the evaluators

---